# When Can We Trust Neural Network Confidence? — Parts 2, 3, 4 on Colab

The data is downloaded **once** into a single shared folder and reused by every part.
*Runtime → Change runtime type → T4 GPU*, then *Run all*.

With `USE_DRIVE = True` the downloads (CIFAR-10, ResNet-18 weights, CIFAR-10-C) and the computed logits are kept
in `MyDrive/nnc_data`, so later sessions skip the download entirely and go straight to the analysis.
Part 1 trains its own models (hours on a GPU) and is not run here.

In [ ]:
REPO_URL = "https://github.com/puppyvn/DL2026-23-23.git"
BRANCH = "test_merge"
USE_DRIVE = False

import os, subprocess
os.chdir("/content")
if not os.path.exists("DL2026-23-23"):
    if subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL]).returncode != 0:
        print(f"Branch {BRANCH!r} not found; cloning the default branch.")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
os.chdir("/content/DL2026-23-23")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["NNC_DATA_DIR"] = "/content/drive/MyDrive/nnc_data"
print("Shared data folder:", os.environ.get("NNC_DATA_DIR", os.path.abspath("downloads")))

## 1. Download everything once (skipped for anything already there)

In [ ]:
!pip -q install "gdown>=5,<6"
# On Drive, delete the 2.9 GB CIFAR-10-C archive after extracting the five corruptions the project uses.
!python -m shared.prepare_data {"--delete-archive" if USE_DRIVE else ""}

## 2. Part 2 — controlled temperature stress test (uses the logits committed in part_2/data)

In [ ]:
%cd /content/DL2026-23-23/part_2
!python data_preparation/prepare_data.py && python evaluation/run_experiment.py && python experiments/run_all.py
%cd /content/DL2026-23-23

## 3. Part 3 — raw confidence under CIFAR-10-C shift

In [ ]:
!python part_3/run_part3.py

## 4. Part 4 — Temperature Scaling and Dirichlet, clean and under shift

In [ ]:
%cd /content/DL2026-23-23/part_4/src
!python experiments.py --part all --no-show --zip
%cd /content/DL2026-23-23

## 5. Download the results

In [ ]:
import glob, shutil
part3 = sorted(glob.glob("part_3/results/part3_*/"))[-1]
part4 = sorted(glob.glob("part_4/src/artifacts/results/*.zip"))[-1]
zips = [shutil.make_archive(part3.rstrip("/"), "zip", part3), part4,
        shutil.make_archive("part_2_results", "zip", "part_2/results")]
print(zips)
from google.colab import files
for z in zips:
    files.download(z)